# Clean viruses → interim

Normalize multi-source virus detections onto modelling keys (spec `04`).

| Item | Detail |
|------|--------|
| Input | `data/01_raw/virus/virus_detections.tsv` + `countries.csv`, `subnational_locations.csv` |
| Output | `data/02_interim/viruses_normalized.parquet`; unmatched regions → `data/02_interim/virus_unmatched_regions.csv` |
| H3 | `H3_RESOLUTION` from `vector_guard.resolve` (locked = 4) |
| Time | `collection_date` → year-month (month required for panel keys) |
| Coords | Locked ladder: exact GPS / backfill → curated centroid → `country_only` |

**Guardrail:** never call external geocoders. Unmatched region strings stay `country_only` and are listed for optional `05` follow-up.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_interim_dir,
    find_repo_root,
    load_virus_detections,
)
from vector_guard.resolve import (
    GEO_PRECISION_CENTROID,
    GEO_PRECISION_COUNTRY,
    GEO_PRECISION_EXACT,
    GEO_PRECISIONS,
    H3_RESOLUTION,
    assign_h3,
    build_name_to_iso2,
    build_subnational_centroid_index,
    load_countries,
    load_subnational_locations,
    lookup_subnational_centroid,
    norm_key,
    parse_year_month_virus,
    region_candidate_keys,
    resolve_coords_ladder,
    resolve_virus_country_iso2,
)

REPO_ROOT = find_repo_root(Path.cwd())
INTERIM_DIR = default_interim_dir(REPO_ROOT)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = INTERIM_DIR / "viruses_normalized.parquet"
UNMATCHED_PATH = INTERIM_DIR / "virus_unmatched_regions.csv"

assert H3_RESOLUTION is not None and H3_RESOLUTION >= 0
REPO_ROOT, OUT_PATH, UNMATCHED_PATH, H3_RESOLUTION

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/02_interim/viruses_normalized.parquet'),
 PosixPath('/workspaces/vector-guard/data/02_interim/virus_unmatched_regions.csv'),
 4)

## Load raw + reference

In [2]:
raw = load_virus_detections()
countries = load_countries()
subnational = load_subnational_locations()

name_to_iso2 = build_name_to_iso2(countries)
centroid_index = build_subnational_centroid_index(subnational)

{
    "raw_rows": len(raw),
    "countries": len(countries),
    "subnational": len(subnational),
    "sources": sorted(raw["source"].fillna("").unique().tolist()),
}

{'raw_rows': 168277,
 'countries': 251,
 'subnational': 7788,
 'sources': ['bvbrc',
  'ecdc_atlas',
  'empres_i',
  'ena_read_run',
  'ena_sequence',
  'pathoplexus']}

## Resolve ISO-2, year-month, coords, H3

Coord priority (locked):

1. Valid `lat`/`lon` when `coord_source` ≠ `none` → `exact_coords`
2. `(country_iso2, subnational_region | nuts_id)` → `subnational_locations.csv` → `subnational_centroid`
3. Else → `country_only`, `h3_cell` null

In [3]:
def _text(value: object) -> str:
    text = str(value or "").strip()
    if text.lower() in {"nan", "none", "null", "na", "n/a", "<na>", "no_value"}:
        return ""
    return text


records = raw.to_dict(orient="records")

country_iso2 = [
    resolve_virus_country_iso2(row.get("country", ""), name_to_iso2) for row in records
]
subnational_region = [_text(row.get("subnational_region", "")) for row in records]
nuts_id = [_text(row.get("nuts_id", "")) for row in records]
coord_source = [_text(row.get("coord_source", "")).lower() for row in records]

year_month = [parse_year_month_virus(row.get("collection_date", "")) for row in records]
year = [pair[0] if pair else pd.NA for pair in year_month]
month = [pair[1] if pair else pd.NA for pair in year_month]

coords = [
    resolve_coords_ladder(
        lat=row.get("lat", ""),
        lon=row.get("lon", ""),
        coord_source=src,
        country_iso2=iso2,
        subnational_region=region,
        nuts_id=nuts,
        centroid_index=centroid_index,
        allow_centroid=True,
    )
    for row, src, iso2, region, nuts in zip(
        records, coord_source, country_iso2, subnational_region, nuts_id, strict=True
    )
]
lat = [item[0] for item in coords]
lon = [item[1] for item in coords]
geo_precision = [item[2] for item in coords]
h3_cell = [assign_h3(la, lo, H3_RESOLUTION) for la, lo in zip(lat, lon, strict=True)]

out = pd.DataFrame(
    {
        "record_id": [_text(row.get("record_id", "")) for row in records],
        "source": [_text(row.get("source", "")) for row in records],
        "pathogen_name": [_text(row.get("pathogen_name", "")) for row in records],
        "vector_group": [_text(row.get("vector_group", "")) for row in records],
        "country_iso2": country_iso2,
        "subnational_region": subnational_region,
        "nuts_id": nuts_id,
        "lat": lat,
        "lon": lon,
        "coord_source": coord_source,
        "geo_precision": geo_precision,
        "h3_cell": h3_cell,
        "year": year,
        "month": month,
        "accession": [_text(row.get("accession", "")) for row in records],
        "host": [_text(row.get("host", "")) for row in records],
    }
)

out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
out["month"] = pd.to_numeric(out["month"], errors="coerce").astype("Int64")

out.head()

,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host
0,ena_sequence:AF206518,ena_sequence,West Nile virus,mosquito,US,"Connecticut, Greenwich-Stanford town line",,41.6486,-72.7594,,subnational_centroid,842a14bffffffff,<NA>,<NA>,AF206518,Culex pipiens
1,ena_sequence:AY187014,ena_sequence,West Nile virus,mosquito,US,"Harris County, TX",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY187014,
2,ena_sequence:AY278442,ena_sequence,West Nile virus,mosquito,RU,"Volgograd, low Volga",,49.5014,44.4488,,subnational_centroid,842d4d3ffffffff,<NA>,<NA>,AY278442,Human
3,ena_sequence:AY428518,ena_sequence,West Nile virus,mosquito,US,"Texas, Wichita county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428518,
4,ena_sequence:AY428519,ena_sequence,West Nile virus,mosquito,US,"Texas, Randall county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428519,


## QA coverage

In [4]:
n = len(out)
tier1 = out["h3_cell"].notna()
qa = {
    "rows": n,
    "iso2_mapped_share": float((out["country_iso2"] != "").mean()),
    "year_month_share": float((out["year"].notna() & out["month"].notna()).mean()),
    "vector_group_share": float((out["vector_group"] != "").mean()),
    "tier1_h3_share": float(tier1.mean()),
    "tier2_country_only_share": float((out["geo_precision"] == GEO_PRECISION_COUNTRY).mean()),
    "exact_among_tier1": float(
        (out.loc[tier1, "geo_precision"] == GEO_PRECISION_EXACT).mean()
    )
    if tier1.any()
    else float("nan"),
    "centroid_among_tier1": float(
        (out.loc[tier1, "geo_precision"] == GEO_PRECISION_CENTROID).mean()
    )
    if tier1.any()
    else float("nan"),
}
display(pd.Series(qa))

print("geo_precision")
display(out["geo_precision"].value_counts(dropna=False))
print("coord_source")
display(out["coord_source"].value_counts(dropna=False))
print("vector_group")
display(out["vector_group"].value_counts(dropna=False))

print("per-source geo")
display(
    out.groupby("source")
    .agg(
        rows=("record_id", "size"),
        iso2=("country_iso2", lambda s: (s != "").mean()),
        tier1=("h3_cell", lambda s: s.notna().mean()),
        ym=("year", lambda s: s.notna().mean()),
    )
    .sort_values("rows", ascending=False)
)

rows                        168277.000000
iso2_mapped_share                0.987550
year_month_share                 0.655104
vector_group_share               1.000000
tier1_h3_share                   0.460419
tier2_country_only_share         0.539581
exact_among_tier1                0.461421
centroid_among_tier1             0.538579
dtype: float64

geo_precision


geo_precision
country_only            90799
subnational_centroid    41728
exact_coords            35750
Name: count, dtype: int64

coord_source


coord_source
                   132527
exact_gps           34187
sample_backfill      1563
Name: count, dtype: int64

vector_group


vector_group
mosquito        110908
biting_midge     42432
tick             14230
sandfly            707
Name: count, dtype: int64

per-source geo


,rows,iso2,tier1,ym
source,,,,
ena_sequence,118381,0.993960,0.330763,0.569906
empres_i,25874,0.998686,1.000000,1.0
ena_read_run,13231,0.917013,0.539566,0.731388
pathoplexus,6377,0.991846,0.564529,0.647954
bvbrc,4108,0.952288,0.342989,0.677702
ecdc_atlas,306,1.000000,0.980392,1.0


## Unmatched subnational / NUTS keys (no geocoding)

Rows with region text or `nuts_id` that did **not** resolve to a centroid and have no exact GPS. Writes rebuildable `data/02_interim/virus_unmatched_regions.csv` with an `in_vectornet_scope` flag so notebook `05` curation can prioritize panel-relevant keys.

In [5]:
unmatched_mask = (
    (out["geo_precision"] == GEO_PRECISION_COUNTRY)
    & (out["country_iso2"] != "")
    & ((out["subnational_region"] != "") | (out["nuts_id"] != ""))
)

unmatched_rows = out.loc[unmatched_mask].copy()

vectornet_iso2 = {
    str(iso).strip().upper()
    for iso, flag in zip(
        countries["iso2"].fillna(""),
        countries["in_vectornet_scope"],
        strict=True,
    )
    if str(iso).strip()
    and str(flag).strip().lower() in {"1", "true", "yes"}
}


def region_lookup_key(row: pd.Series) -> str:
    """Display/report key aligned with runtime lookup candidates."""
    if row["nuts_id"]:
        return norm_key(row["nuts_id"])
    keys = region_candidate_keys(row["subnational_region"])
    return keys[0] if keys else ""


unmatched_rows["region_key"] = unmatched_rows.apply(region_lookup_key, axis=1)
# Confirm still unmatched under the same lookup used by the ladder
still_unmatched = []
for row in unmatched_rows.itertuples(index=False):
    hit = lookup_subnational_centroid(
        row.country_iso2,
        subnational_region=row.subnational_region,
        nuts_id=row.nuts_id,
        centroid_index=centroid_index,
    )
    if hit is None and row.region_key:
        still_unmatched.append(
            {
                "country_iso2": row.country_iso2,
                "region_key": row.region_key,
                "subnational_region": row.subnational_region,
                "nuts_id": row.nuts_id,
                "source": row.source,
            }
        )

unmatched_cols = [
    "country_iso2",
    "region_key",
    "subnational_region",
    "nuts_id",
    "source",
    "rows",
    "in_vectornet_scope",
]
unmatched = (
    pd.DataFrame(still_unmatched)
    .value_counts(
        ["country_iso2", "region_key", "subnational_region", "nuts_id", "source"]
    )
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
    if still_unmatched
    else pd.DataFrame(columns=unmatched_cols[:-1])
)
unmatched["in_vectornet_scope"] = unmatched["country_iso2"].isin(vectornet_iso2).astype(int)
unmatched = unmatched[unmatched_cols]

unmatched.to_csv(UNMATCHED_PATH, index=False)
vn_keys = int(unmatched["in_vectornet_scope"].sum())
vn_rows = int(unmatched.loc[unmatched["in_vectornet_scope"] == 1, "rows"].sum())
print(f"Wrote {UNMATCHED_PATH}")
print(
    f"Unmatched region/NUTS keys: {len(unmatched):,} "
    f"(VectorNet-scope keys={vn_keys:,}, rows={vn_rows:,})"
)
display(unmatched.head(40))
display(
    unmatched.loc[unmatched["in_vectornet_scope"] == 1]
    .head(20)
    .reset_index(drop=True)
)

Wrote /workspaces/vector-guard/data/02_interim/virus_unmatched_regions.csv
Unmatched region/NUTS keys: 1,875 (VectorNet-scope keys=628, rows=2,071)


,country_iso2,region_key,subnational_region,nuts_id,source,rows,in_vectornet_scope
0,VN,south,South,,ena_sequence,1216,0
1,CN,guangzhou,Guangzhou,,ena_sequence,674,0
2,VN,nha-trang,Nha Trang,,ena_sequence,562,0
3,CN,xi-shuang-ban-na,Xi Shuang Ban Na,,ena_sequence,259,0
4,VN,south,south,,ena_sequence,240,0
5,CN,hangzhou,Hangzhou,,ena_sequence,171,0
6,KE,mombasa,Mombasa,,ena_sequence,126,0
7,ID,surabaya,Surabaya,,ena_sequence,120,0
8,KH,kch,KCH,,ena_sequence,118,0
9,SA,jeddah,Jeddah,,ena_sequence,104,0


,country_iso2,region_key,subnational_region,nuts_id,source,rows,in_vectornet_scope
0,GR,rhodes,Rhodes,,ena_sequence,33,1
1,MA,azrou,Azrou,,ena_sequence,30,1
2,RS,tutin,Tutin,,ena_sequence,30,1
3,TN,ghardimaou,Ghardimaou,,ena_sequence,30,1
4,IL,jordan-valley,Jordan Valley,,ena_sequence,30,1
5,AT,vienna-international-airport,Vienna International Airport,,ena_sequence,22,1
6,XK,malisheve,Malisheve,,ena_sequence,21,1
7,IL,yokneam,Yokneam,,ena_sequence,21,1
8,PT,imported-unknown-origin,"imported, unknown origin",,ena_sequence,20,1
9,GR,aghia-paraskevi-lesvos,Aghia Paraskevi-Lesvos,,ena_sequence,20,1


## Assert + write parquet

In [6]:
required_cols = [
    "record_id",
    "source",
    "pathogen_name",
    "vector_group",
    "country_iso2",
    "subnational_region",
    "nuts_id",
    "lat",
    "lon",
    "coord_source",
    "geo_precision",
    "h3_cell",
    "year",
    "month",
    "accession",
    "host",
]
assert list(out.columns) == required_cols
assert out["geo_precision"].isin(GEO_PRECISIONS).all()
assert out["record_id"].ne("").all()

has_coords = out["lat"].notna() & out["lon"].notna()
assert out.loc[has_coords, "lat"].between(-90, 90).all()
assert out.loc[has_coords, "lon"].between(-180, 180).all()
assert (
    out.loc[has_coords, "geo_precision"]
    .isin({GEO_PRECISION_EXACT, GEO_PRECISION_CENTROID})
    .all()
)
assert out.loc[has_coords, "h3_cell"].notna().all()
assert out.loc[~has_coords, "h3_cell"].isna().all()
assert out.loc[~has_coords, "geo_precision"].eq(GEO_PRECISION_COUNTRY).all()
assert (out.loc[out["month"].notna(), "month"].between(1, 12)).all()

# Exact GPS must win over centroid when both would apply
exact_gps_rows = out["coord_source"].isin({"exact_gps", "sample_backfill"}) & has_coords
assert out.loc[exact_gps_rows, "geo_precision"].eq(GEO_PRECISION_EXACT).all()

out.to_parquet(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(out):,} rows)")
display(out["geo_precision"].value_counts())
display(out.head(10))
display(out.tail(5))

Wrote /workspaces/vector-guard/data/02_interim/viruses_normalized.parquet (168,277 rows)


geo_precision
country_only            90799
subnational_centroid    41728
exact_coords            35750
Name: count, dtype: int64

,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host
0,ena_sequence:AF206518,ena_sequence,West Nile virus,mosquito,US,"Connecticut, Greenwich-Stanford town line",,41.6486,-72.7594,,subnational_centroid,842a14bffffffff,<NA>,<NA>,AF206518,Culex pipiens
1,ena_sequence:AY187014,ena_sequence,West Nile virus,mosquito,US,"Harris County, TX",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY187014,
2,ena_sequence:AY278442,ena_sequence,West Nile virus,mosquito,RU,"Volgograd, low Volga",,49.5014,44.4488,,subnational_centroid,842d4d3ffffffff,<NA>,<NA>,AY278442,Human
3,ena_sequence:AY428518,ena_sequence,West Nile virus,mosquito,US,"Texas, Wichita county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428518,
4,ena_sequence:AY428519,ena_sequence,West Nile virus,mosquito,US,"Texas, Randall county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428519,
5,ena_sequence:AY428522,ena_sequence,West Nile virus,mosquito,US,Illinois,,39.9460,-89.1991,,subnational_centroid,8426627ffffffff,<NA>,<NA>,AY428522,
6,ena_sequence:AY639639,ena_sequence,West Nile virus,mosquito,IN,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,AY639639,Culex tritainiorhynchus
7,ena_sequence:AY701413,ena_sequence,West Nile virus,mosquito,MA,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,AY701413,horse
8,ena_sequence:AY712948,ena_sequence,West Nile virus,mosquito,US,Texas,,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY712948,Culex quinquefasciatus
9,ena_sequence:DQ010345,ena_sequence,West Nile virus,mosquito,US,"Suffolk Couunty, NY",,43.1988,-75.3242,,subnational_centroid,842b887ffffffff,<NA>,<NA>,DQ010345,American crow


,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host
168272,ecdc_atlas:monthly:ES533:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Menorca,ES533,39.950076,4.129265,,subnational_centroid,8439433ffffffff,2026,10,,equid_outbreaks:1
168273,ecdc_atlas:monthly:ES611:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Almería,ES611,37.210875,-2.312152,,subnational_centroid,8438243ffffffff,2026,10,,equid_outbreaks:1
168274,ecdc_atlas:monthly:ES616:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Jaén,ES616,38.017816,-3.444921,,subnational_centroid,84390a1ffffffff,2026,10,,equid_outbreaks:1;bird_outbreaks:1
168275,ecdc_atlas:monthly:ES705:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Gran Canaria,ES705,27.958743,-15.585429,,subnational_centroid,84344cbffffffff,2026,10,,equid_outbreaks:3
168276,ecdc_atlas:monthly:CH033:2026-W41,ecdc_atlas,West Nile virus,mosquito,CH,Aargau,CH033,47.411178,8.154324,,subnational_centroid,841f83bffffffff,2026,10,,equid_outbreaks:1
